## Outline

1. Install Unsloth and the supporting libraries.
2. Set a few simple training constants.
3. Load the MedQuad dataset, filter weak rows, and keep a small quick-run subset.
4. Load Gemma 4 E4B with Unsloth in 4-bit mode and attach the LoRA adapters.
5. Generate three held-out sample answers before fine-tuning.
6. Fine-tune with QLoRA (Unsloth + TRL `SFTTrainer`).
7. Compare the three before/after responses and save the adapter.

## 1. Install the required libraries

Unsloth replaces the manual `bitsandbytes` + `peft` setup: it loads the 4-bit model, patches it for faster training and lower VRAM, and wraps the LoRA creation.
If the model fails to load with a "model type not recognized" style error, upgrade `transformers` (`%pip install -q -U transformers`) and restart the runtime.

In [1]:
#%pip install -q -U unsloth unsloth_zoo
#%pip install -q -U datasets sentencepiece

## 2. Optional Hugging Face login

If the model download is gated in your environment or you want to push the adapter later, set `HF_TOKEN` in the notebook environment before running this cell.

In [2]:
import os
from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face.")
else:
    print("HF_TOKEN not found. Continuing without login.")

HF_TOKEN not found. Continuing without login.


## 3. Imports and configuration

`unsloth` must be imported **before** `transformers`, `trl` and `peft` so its patches are applied.
Keep the configuration small first. You can scale the dataset size, epochs, or LoRA rank after the basic run works.

In [3]:
from unsloth import FastModel, is_bfloat16_supported  # keep this import first

import os
import random
import re

import torch
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from transformers import set_seed

MODEL_ID = "unsloth/gemma-4-E4B-it"
DATASET_ID = "keivalya/MedQuad-MedicalQnADataset"
SAMPLE_SIZE = 240
EVAL_SIZE = 24
COMPARISON_SAMPLE_COUNT = 3
MAX_SEQ_LENGTH = 768
LOAD_IN_4BIT = True
OUTPUT_DIR = "gemma4-e4b-medquad-unsloth"
SEED = 42

set_seed(SEED)
random.seed(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(
        f"GPU memory (GB): {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}"
    )
    print(f"bf16 supported: {is_bfloat16_supported()}")

ImportError: DLL load failed while importing _C: Uma política de Controle de Aplicativo bloqueou este arquivo.

## 4. Load and inspect the dataset

The MedQuad dataset is a medical question-answering dataset on Hugging Face. For faster iteration, we filter out obviously weak rows, keep a small subset, and then split it into train and evaluation sets.

In [ ]:
raw_dataset = load_dataset(DATASET_ID, split="train")
print(raw_dataset)
print("Columns:", raw_dataset.column_names)
print("First example:", raw_dataset[0])


def clean_text(text):
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text


BAD_ANSWER_PREFIXES = (
    "These resources address",
    "These resources from MedlinePlus",
    "For more information, go to",
    "For more information go to",
)


def keep_example(example):
    question = clean_text(example["Question"])
    answer = clean_text(example["Answer"])

    if len(question) < 12 or len(answer) < 40:
        return False
    if len(answer) > 900:
        return False
    if any(answer.startswith(prefix) for prefix in BAD_ANSWER_PREFIXES):
        return False
    return True


filtered_dataset = raw_dataset.filter(keep_example)
subset = filtered_dataset.shuffle(seed=SEED).select(range(SAMPLE_SIZE))
split_dataset = subset.train_test_split(test_size=EVAL_SIZE, seed=SEED)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Filtered dataset size: {len(filtered_dataset)}")
print(f"Train size: {len(train_dataset)}")
print(f"Eval size: {len(eval_dataset)}")

## 5. Load Gemma 4 E4B with Unsloth and attach LoRA

`FastModel.from_pretrained` loads the quantized base model and tokenizer in one call (no `BitsAndBytesConfig` needed).
`FastModel.get_peft_model` then adds the LoRA adapters. Since LoRA's `B` matrices start at zero, the model still behaves exactly like the base model, so the "before fine-tuning" answers below are the true baseline.

In [ ]:
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    load_in_8bit=False,
    full_finetuning=False,
    dtype=None,  # auto: bf16 when supported, otherwise fp16 (e.g. T4)
)

# Some Unsloth models return a multimodal Processor; its text tokenizer lives in `.tokenizer`.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
if text_tokenizer.pad_token is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,  # text-only QA task
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=8,
    lora_alpha=16,
    lora_dropout=0,  # 0 keeps Unsloth's fast path enabled
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

model.print_trainable_parameters()
print("Gemma 4 E4B loaded with Unsloth and LoRA attached.")

## 6. Prompt formatting and evaluation helpers

We turn each row into a chat-style instruction with a short system prompt and a direct user question.
The tokenizer's own chat template is used, and the leading BOS token is stripped from the rendered text because the tokenizer adds it again when tokenizing.

In [ ]:
SYSTEM_PROMPT = "Answer the medical question directly in 2-4 factual sentences."


def apply_chat_template(messages, add_generation_prompt=False):
    kwargs = {
        "tokenize": False,
        "add_generation_prompt": add_generation_prompt,
    }
    try:
        text = text_tokenizer.apply_chat_template(
            messages,
            enable_thinking=False,  # only used by templates that support it (Qwen)
            **kwargs,
        )
    except TypeError:
        text = text_tokenizer.apply_chat_template(
            messages,
            chat_template_kwargs={"enable_thinking": False},
            **kwargs,
        )
    return strip_bos(text)


def strip_bos(text):
    bos = text_tokenizer.bos_token
    if bos and text.startswith(bos):
        text = text[len(bos) :]
    return text


def clean_answer_text(answer):
    answer = clean_text(answer)
    answer = re.sub(r"^answer\s*:\s*", "", answer, flags=re.IGNORECASE)
    return answer


def make_user_prompt(question):
    return (
        f"Question: {clean_text(question)}\n\n"
        "Respond with only the answer. Do not restate the question."
    )


def format_training_example(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": make_user_prompt(example["Question"]),
        },
        {
            "role": "assistant",
            "content": clean_answer_text(example["Answer"]),
        },
    ]
    return {"text": apply_chat_template(messages, add_generation_prompt=False)}


@torch.inference_mode()
def generate_answer(question, max_new_tokens=160):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": make_user_prompt(question),
        },
    ]
    prompt = apply_chat_template(messages, add_generation_prompt=True)
    model_inputs = text_tokenizer(prompt, return_tensors="pt").to(model.device)

    # eos_token_id is intentionally not overridden: the model's generation_config
    # already lists the end-of-turn token(s) for each family.
    generated = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
        pad_token_id=text_tokenizer.pad_token_id,
    )

    new_tokens = generated[0][model_inputs["input_ids"].shape[1] :]
    text = text_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return clean_answer_text(re.sub(r"<think>.*?</think>", " ", text, flags=re.DOTALL))


def clip_text(text, max_chars=600):
    text = re.sub(r"<think>.*?</think>", " ", text, flags=re.DOTALL)
    text = re.sub(r"\s+", " ", text).strip()
    return text if len(text) <= max_chars else text[:max_chars] + "..."


def generate_preview_rows(dataset, sample_count=COMPARISON_SAMPLE_COUNT):
    sample_count = min(sample_count, len(dataset))
    subset = dataset.select(range(sample_count))
    previews = []

    for row in subset:
        previews.append(
            {
                "question": row["Question"],
                "prediction": generate_answer(row["Question"]),
                "reference": clean_answer_text(row["Answer"]),
            }
        )

    return previews


formatted_train_dataset = train_dataset.map(
    format_training_example,
    remove_columns=train_dataset.column_names,
)

comparison_examples = eval_dataset.shuffle(seed=SEED).select(
    range(COMPARISON_SAMPLE_COUNT)
)

print(formatted_train_dataset[0])

## 7. Generate three sample answers before fine-tuning

Instead of numeric evaluation, this notebook uses the same three held-out examples before and after training.
That keeps the review simple and makes it easy to compare the answer style directly.

In [ ]:
FastModel.for_inference(model)  # Unsloth's 2x faster inference mode

baseline_previews = generate_preview_rows(comparison_examples)

for idx, sample in enumerate(baseline_previews, start=1):
    print(f"\n--- Preview {idx} ---")
    print("Question:", sample["question"])
    print("Model answer (before fine-tuning):", clip_text(sample["prediction"]))
    print("Reference answer:", clip_text(sample["reference"]))

## 8. Configure the trainer

The LoRA adapters were already attached in section 5, so only the `SFTTrainer` setup remains. Unsloth handles gradient checkpointing, and `adamw_8bit` replaces `paged_adamw_8bit`.

In [ ]:
FastModel.for_training(model)

trainer_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    learning_rate=1e-4,
    warmup_steps=5,
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
    packing=False,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=SEED,
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_train_dataset,
    args=trainer_config,
    processing_class=text_tokenizer,
)

model.print_trainable_parameters()

## 9. Train the adapter

This is a quick-run training pass over a small subset.

In [ ]:
train_result = trainer.train()
train_result

## 10. Save the adapter

This writes the LoRA adapter and tokenizer locally so you can reload them later or push them to the Hub.

In [ ]:
model.save_pretrained(OUTPUT_DIR)
text_tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved adapter to: {OUTPUT_DIR}")

In [ ]:
PUSH_TO_HUB = False  # set to True after logging in with a write token
HF_REPO_ID = "YOUR_HF_USERNAME/gemma4-e4b-medquad-unsloth"

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID)
    text_tokenizer.push_to_hub(HF_REPO_ID)

## 11. Generate the same three sample answers after fine-tuning

This runs the exact same three held-out prompts again after training.

In [ ]:
FastModel.for_inference(model)

fine_tuned_previews = generate_preview_rows(comparison_examples)

for idx, sample in enumerate(fine_tuned_previews, start=1):
    print(f"\n--- Preview {idx} ---")
    print("Question:", sample["question"])
    print("Model answer (after fine-tuning):", clip_text(sample["prediction"]))
    print("Reference answer:", clip_text(sample["reference"]))

## 12. Compare the before and after responses

In [ ]:
for idx, (before, after) in enumerate(
    zip(baseline_previews, fine_tuned_previews), start=1
):
    print(f"\n{'=' * 80}")
    print(f"Sample {idx}")
    print(f"{'=' * 80}")
    print("Question:", before["question"])
    print("\nBefore fine-tuning:")
    print(clip_text(before["prediction"]))
    print("\nAfter fine-tuning:")
    print(clip_text(after["prediction"]))
    print("\nReference answer:")
    print(clip_text(before["reference"]))

## 13. Extended experiment: zero-shot vs 3x vs 5x (3 epochs each)

This block is **separate** from the quick run above. It compares three conditions on the same held-out questions, always with a zero-shot prompt (system prompt + question, no examples in the context):

- **zero-shot**: the base model, no fine-tuning.
- **3x**: fresh LoRA trained for **3 epochs** on 3x the base training set (`3 * len(train_dataset)` examples).
- **5x**: fresh LoRA trained for **3 epochs** on 5x the base training set (the 3x examples are a subset of the 5x ones).

For each condition it measures generation speed (tokens/s, time to first token, latency), answer quality against the reference (token F1, ROUGE-L, answer NLL / perplexity), peak VRAM and training cost, then saves CSVs and plots.
Each condition starts from a freshly loaded model, so the model from section 5 is freed first. Run sections 1-6 before this block; sections 7-12 can be skipped.

### 13.1 Configuration, regimes and data

The extra training rows come from the same shuffled pool as the base subset, so they never overlap with the base training rows or the held-out eval rows.

In [ ]:
import gc
import math
import time
from collections import Counter

import pandas as pd
import torch.nn.functional as F
from datasets import concatenate_datasets
from IPython.display import display

EXTENDED_EPOCHS = 3
REGIMES = [("zero-shot", 0), ("3x", 3), ("5x", 5)]  # (name, multiplier of the base train set)
EXTENDED_EVAL_N = 24  # held-out questions used for the metrics
GEN_MAX_NEW_TOKENS = 160
EXTENDED_DIR = f"{OUTPUT_DIR}-extended"
os.makedirs(EXTENDED_DIR, exist_ok=True)

# Free the model/trainer from the quick run so each regime starts from a fresh model.
for _name in ("trainer", "model", "train_result"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

# Extra training rows: same shuffle as the base subset, so they never overlap with it
# (and therefore never overlap with the held-out eval set).
BASE_TRAIN_SIZE = len(train_dataset)
max_multiplier = max(mult for _, mult in REGIMES)
extra_needed = (max_multiplier - 1) * BASE_TRAIN_SIZE
shuffled_filtered = filtered_dataset.shuffle(seed=SEED)
assert len(shuffled_filtered) >= SAMPLE_SIZE + extra_needed, "Not enough filtered rows for the largest regime."
extra_pool = shuffled_filtered.select(range(SAMPLE_SIZE, SAMPLE_SIZE + extra_needed))


def build_train_subset(multiplier):
    extra = extra_pool.select(range((multiplier - 1) * BASE_TRAIN_SIZE))
    return concatenate_datasets([train_dataset, extra])


extended_eval = eval_dataset.select(range(min(EXTENDED_EVAL_N, len(eval_dataset))))

for name, mult in REGIMES:
    size = BASE_TRAIN_SIZE * mult
    print(f"{name:>10}: {size} training examples x {EXTENDED_EPOCHS} epochs" if mult else f"{name:>10}: no training")
print(f"Eval questions: {len(extended_eval)} | max_new_tokens: {GEN_MAX_NEW_TOKENS}")

### 13.2 Model builder and metric helpers

In [ ]:
def build_model():
    """Load a fresh 4-bit base model with new LoRA adapters (same settings as section 5)."""
    global text_tokenizer
    new_model, new_tokenizer = FastModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        load_in_8bit=False,
        full_finetuning=False,
        dtype=None,
    )
    text_tokenizer = getattr(new_tokenizer, "tokenizer", new_tokenizer)
    if text_tokenizer.pad_token is None:
        text_tokenizer.pad_token = text_tokenizer.eos_token

    new_model = FastModel.get_peft_model(
        new_model,
        finetune_vision_layers=False,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=8,
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )
    return new_model

In [ ]:
def word_tokens(text):
    return re.findall(r"\w+", str(text).lower())


def token_f1(prediction, reference):
    pred, ref = word_tokens(prediction), word_tokens(reference)
    if not pred or not ref:
        return 0.0
    overlap = sum((Counter(pred) & Counter(ref)).values())
    if overlap == 0:
        return 0.0
    precision, recall = overlap / len(pred), overlap / len(ref)
    return 2 * precision * recall / (precision + recall)


def rouge_l_f1(prediction, reference):
    pred, ref = word_tokens(prediction), word_tokens(reference)
    if not pred or not ref:
        return 0.0
    previous = [0] * (len(ref) + 1)
    for i in range(1, len(pred) + 1):
        current = [0] * (len(ref) + 1)
        for j in range(1, len(ref) + 1):
            if pred[i - 1] == ref[j - 1]:
                current[j] = previous[j - 1] + 1
            else:
                current[j] = max(previous[j], current[j - 1])
        previous = current
    lcs = previous[-1]
    if lcs == 0:
        return 0.0
    precision, recall = lcs / len(pred), lcs / len(ref)
    return 2 * precision * recall / (precision + recall)

In [ ]:
def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def postprocess_answer(text):
    text = re.sub(r"<think>.*?</think>", " ", text, flags=re.DOTALL)
    return clean_answer_text(text)


def build_prompt(question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(question)},
    ]
    return messages, apply_chat_template(messages, add_generation_prompt=True)


@torch.inference_mode()
def timed_generate(model, question, max_new_tokens):
    _, prompt = build_prompt(question)
    inputs = text_tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    gen_kwargs = dict(do_sample=False, use_cache=True, pad_token_id=text_tokenizer.pad_token_id)

    # Time to first token: generate a single token.
    cuda_sync()
    start = time.perf_counter()
    model.generate(**inputs, max_new_tokens=1, **gen_kwargs)
    cuda_sync()
    ttft = time.perf_counter() - start

    # Full answer.
    cuda_sync()
    start = time.perf_counter()
    generated = model.generate(**inputs, max_new_tokens=max_new_tokens, **gen_kwargs)
    cuda_sync()
    total = time.perf_counter() - start

    new_tokens = generated[0][prompt_len:]
    n_new = int(new_tokens.shape[0])
    raw_text = text_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    decode_time = total - ttft
    return {
        "prediction": postprocess_answer(raw_text),
        "prompt_tokens": prompt_len,
        "new_tokens": n_new,
        "latency_s": total,
        "ttft_s": ttft,
        "tokens_per_s": n_new / total if total > 0 else float("nan"),
        "decode_tokens_per_s": (n_new - 1) / decode_time if n_new > 1 and decode_time > 0 else float("nan"),
        "finished": n_new < max_new_tokens,  # False -> hit the token limit before the end-of-turn token
    }


@torch.inference_mode()
def answer_nll(model, question, reference):
    """Sum of the negative log-likelihood of the reference answer tokens, and the token count."""
    messages, prompt = build_prompt(question)
    full = apply_chat_template(
        messages + [{"role": "assistant", "content": reference}],
        add_generation_prompt=False,
    )
    n_prompt = len(text_tokenizer(prompt)["input_ids"])
    encoded = text_tokenizer(
        full, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH
    ).to(model.device)
    input_ids = encoded["input_ids"]
    if input_ids.shape[1] <= n_prompt + 1:
        return 0.0, 0
    logits = model(**encoded).logits[0, n_prompt - 1 : -1].float()
    targets = input_ids[0, n_prompt:]
    nll = F.cross_entropy(logits, targets, reduction="sum")
    return nll.item(), int(targets.numel())


def evaluate_model(model, regime):
    FastModel.for_inference(model)
    timed_generate(model, extended_eval[0]["Question"], max_new_tokens=8)  # warm-up, not measured

    torch.cuda.reset_peak_memory_stats()
    rows = []
    for row in extended_eval:
        reference = clean_answer_text(row["Answer"])
        result = timed_generate(model, row["Question"], GEN_MAX_NEW_TOKENS)
        nll_sum, nll_tokens = answer_nll(model, row["Question"], reference)
        rows.append(
            {
                "regime": regime,
                "question": row["Question"],
                "reference": reference,
                **result,
                "token_f1": token_f1(result["prediction"], reference),
                "rouge_l": rouge_l_f1(result["prediction"], reference),
                "answer_words": len(word_tokens(result["prediction"])),
                "reference_words": len(word_tokens(reference)),
                "nll_sum": nll_sum,
                "nll_tokens": nll_tokens,
            }
        )
    peak_vram_gb = torch.cuda.max_memory_allocated() / 1024**3
    return rows, peak_vram_gb

In [ ]:
def train_regime(model, train_ds, epochs, tag):
    formatted = train_ds.map(format_training_example, remove_columns=train_ds.column_names)
    token_lists = text_tokenizer(
        formatted["text"], truncation=True, max_length=MAX_SEQ_LENGTH
    )["input_ids"]
    tokens_per_epoch = sum(len(ids) for ids in token_lists)

    config = SFTConfig(
        output_dir=f"{EXTENDED_DIR}/trainer-{tag}",
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        num_train_epochs=epochs,
        learning_rate=1e-4,
        warmup_steps=5,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        packing=False,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        bf16=is_bfloat16_supported(),
        fp16=not is_bfloat16_supported(),
    )

    FastModel.for_training(model)
    trainer = SFTTrainer(
        model=model,
        train_dataset=formatted,
        args=config,
        processing_class=text_tokenizer,
    )

    torch.cuda.reset_peak_memory_stats()
    result = trainer.train()
    cuda_sync()
    peak_vram_gb = torch.cuda.max_memory_allocated() / 1024**3

    runtime = result.metrics.get("train_runtime", float("nan"))
    info = {
        "train_examples": len(train_ds),
        "epochs": epochs,
        "train_runtime_min": runtime / 60,
        "train_loss": result.training_loss,
        "train_samples_per_s": result.metrics.get("train_samples_per_second", float("nan")),
        "train_tokens_per_s": tokens_per_epoch * epochs / runtime if runtime else float("nan"),
        "train_peak_vram_gb": peak_vram_gb,
    }
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    return info

### 13.3 Run the three regimes

For each regime: load a fresh model, train (3x and 5x only, 3 epochs), then measure generation speed and quality on the same held-out questions. Partial results are saved to CSV after each regime.

In [ ]:
all_rows = []
summary_rows = []

for regime, multiplier in REGIMES:
    print(f"\n{'=' * 80}\nRegime: {regime}\n{'=' * 80}")
    model = build_model()

    train_info = {}
    if multiplier > 0:
        train_info = train_regime(
            model, build_train_subset(multiplier), EXTENDED_EPOCHS, regime
        )
        model.save_pretrained(f"{EXTENDED_DIR}/adapter-{regime}")

    rows, gen_peak_vram_gb = evaluate_model(model, regime)
    all_rows.extend(rows)

    df_regime = pd.DataFrame(rows)
    total_new_tokens = df_regime["new_tokens"].sum()
    total_time = df_regime["latency_s"].sum()
    summary_rows.append(
        {
            "regime": regime,
            **train_info,
            "gen_tokens_per_s": total_new_tokens / total_time,
            "decode_tokens_per_s": df_regime["decode_tokens_per_s"].mean(),
            "ttft_s": df_regime["ttft_s"].mean(),
            "latency_s": df_regime["latency_s"].mean(),
            "avg_new_tokens": df_regime["new_tokens"].mean(),
            "finished_rate": df_regime["finished"].mean(),
            "token_f1": df_regime["token_f1"].mean(),
            "rouge_l": df_regime["rouge_l"].mean(),
            "answer_nll": df_regime["nll_sum"].sum() / max(df_regime["nll_tokens"].sum(), 1),
            "answer_words": df_regime["answer_words"].mean(),
            "reference_words": df_regime["reference_words"].mean(),
            "gen_peak_vram_gb": gen_peak_vram_gb,
        }
    )
    summary_rows[-1]["answer_ppl"] = math.exp(summary_rows[-1]["answer_nll"])

    # Save partial results after every regime so a crash does not lose them.
    pd.DataFrame(all_rows).to_csv(f"{EXTENDED_DIR}/answers_per_question.csv", index=False)
    pd.DataFrame(summary_rows).to_csv(f"{EXTENDED_DIR}/metrics_summary.csv", index=False)

    del model
    gc.collect()
    torch.cuda.empty_cache()

### 13.4 Summary table

In [ ]:
results_df = pd.DataFrame(all_rows)
summary_df = pd.DataFrame(summary_rows).set_index("regime")

print("Summary (one column per regime):")
display(summary_df.T)
print(f"Saved: {EXTENDED_DIR}/metrics_summary.csv and {EXTENDED_DIR}/answers_per_question.csv")

### 13.5 Plots

Each metric has its own figure.

In [ ]:
import matplotlib.pyplot as plt


def bar_plot(columns, title, ylabel, labels=None):
    data = summary_df[columns]
    ax = data.plot(kind="bar", figsize=(7, 4), rot=0)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    if labels:
        ax.legend(labels)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

#### Plot 1 - Generation throughput (tokens/s)

In [ ]:
bar_plot(["gen_tokens_per_s", "decode_tokens_per_s"], "Generation speed", "tokens / second", ["overall", "decode only"])

#### Plot 2 - Time to first token and latency (s)

In [ ]:
bar_plot(["ttft_s", "latency_s"], "Latency per answer", "seconds", ["time to first token", "full answer"])

#### Plot 3 - Answer quality vs reference

In [ ]:
bar_plot(["token_f1", "rouge_l"], "Overlap with the reference answer", "score (0-1)", ["token F1", "ROUGE-L F1"])

#### Plot 4 - Answer NLL (lower is better)

In [ ]:
bar_plot(["answer_nll"], "Negative log-likelihood of the reference answer tokens", "nats / token")

#### Plot 5 - Training cost (3x and 5x)

In [ ]:
bar_plot(["train_runtime_min"], "Training time (3 epochs)", "minutes")

#### Plot 6 - Peak VRAM (GB)

In [ ]:
bar_plot(["gen_peak_vram_gb"], "Peak GPU memory during generation", "GB")

### 13.6 Side-by-side answers

In [ ]:
N_SHOW = min(3, len(extended_eval))

for idx in range(N_SHOW):
    question = extended_eval[idx]["Question"]
    print(f"\n{'=' * 80}\nSample {idx + 1}\n{'=' * 80}")
    print("Question:", question)
    for regime, _ in REGIMES:
        answer = results_df[(results_df["regime"] == regime) & (results_df["question"] == question)]["prediction"].iloc[0]
        print(f"\n[{regime}]")
        print(clip_text(answer))
    print("\n[reference]")
    print(clip_text(clean_answer_text(extended_eval[idx]["Answer"])))

## 14. Speed and precision benchmarks (base vs fine-tuned)

This block is **separate** from the sections above. It measures the same loaded model twice: **base** (LoRA adapter disabled) and **fine-tuned** (adapter on).
Section 13 frees the model, so this block reloads the quick-run adapter saved in section 10 (`OUTPUT_DIR`). Run sections 1-6 and 9-10 before this block.

**Speed** (MedQuAD held-out questions, greedy decoding, batch size 1):

| Metric | What it measures | Better |
|---|---|---|
| TTFT (time to first token) | delay from the request until the first new token comes out (tokenization + prefill + 1st token) | lower |
| Tokens per second | generation speed after the first token: `(new_tokens - 1) / (TTLT - TTFT)` | higher |
| Total response time | whole call: tokenize + generate + decode + clean the answer | lower |
| TTLT (time to last token) | delay from the request until the last token comes out | lower |

**Precision** (accuracy, 2 benchmarks):

| Benchmark | Dataset | Task | Chance |
|---|---|---|---|
| MedQA (USMLE) | `GBaker/MedQA-USMLE-4-options`, test split | clinical exam question, options A-D | 25% |
| PubMedQA | `qiaojin/PubMedQA`, `pqa_labeled` | abstract + research question, yes / no / maybe | 33% |

MedQA tests clinical knowledge and PubMedQA tests reading biomedical evidence, so the two cover different skills. Both are in the MedGemma and HuatuoGPT-o1 reports and in the Open Medical-LLM Leaderboard, so the numbers can be compared with published ones.
The answer is picked by the highest next-token score among the option letters (same idea as `lm-evaluation-harness`), so reasoning or verbose models are scored the same way and nothing depends on parsing text.
Each metric is shown with a 95% confidence interval (`mean +- 1.96 * std / sqrt(n)`).

### 14.1 Configuration and model

Keep the sizes small on a T4. Increase `MEDQA_N` / `PUBMEDQA_N` for tighter intervals (100 questions -> about +-9 points).

In [ ]:
import json
import time
from contextlib import nullcontext

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from transformers.generation.streamers import BaseStreamer

MODEL_NAME = "Gemma 4 E4B"
SPEED_N = 20  # MedQuAD held-out questions for the speed test
SPEED_MAX_NEW_TOKENS = 160
MEDQA_N = 100
PUBMEDQA_N = 100
CONDITIONS = ["base", "fine-tuned"]
BENCH_DIR = f"{OUTPUT_DIR}-benchmark"
os.makedirs(BENCH_DIR, exist_ok=True)

if "model" not in globals():
    model, tokenizer = FastModel.from_pretrained(
        model_name=OUTPUT_DIR,  # LoRA adapter saved in section 10
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        load_in_8bit=False,
        full_finetuning=False,
        dtype=None,
    )
    text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
    if text_tokenizer.pad_token is None:
        text_tokenizer.pad_token = text_tokenizer.eos_token

FastModel.for_inference(model)


def adapter_mode(condition):
    # "base" = same weights with the LoRA adapter turned off.
    return model.disable_adapter() if condition == "base" else nullcontext()


def mean_ci(values):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    mean = values.mean()
    half = 1.96 * values.std(ddof=1) / np.sqrt(len(values)) if len(values) > 1 else 0.0
    return mean, mean - half, mean + half


print(f"Model: {MODEL_NAME} | results in: {BENCH_DIR}")

### 14.2 Speed test: TTFT, tokens/s, total response time and TTLT

`TokenTimer` is a streamer: `generate()` calls `put()` every time a new token is ready, so we save the clock at each token.
The clock starts before tokenizing the prompt. One warm-up answer per condition is not measured.

In [ ]:
class TokenTimer(BaseStreamer):
    def __init__(self):
        self.times = []
        self.is_prompt = True

    def put(self, value):
        if self.is_prompt:  # the first call is the prompt, not a new token
            self.is_prompt = False
            return
        self.times.append(time.perf_counter())

    def end(self):
        pass


@torch.inference_mode()
def timed_answer(question, max_new_tokens=SPEED_MAX_NEW_TOKENS):
    start = time.perf_counter()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(question)},
    ]
    prompt = apply_chat_template(messages, add_generation_prompt=True)
    model_inputs = text_tokenizer(prompt, return_tensors="pt").to(model.device)

    timer = TokenTimer()
    generated = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
        pad_token_id=text_tokenizer.pad_token_id,
        streamer=timer,
    )
    new_tokens = generated[0][model_inputs["input_ids"].shape[1] :]
    text = text_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    answer = clean_answer_text(re.sub(r"<think>.*?</think>", " ", text, flags=re.DOTALL))
    end = time.perf_counter()

    n_new = len(timer.times)
    ttft = timer.times[0] - start
    ttlt = timer.times[-1] - start
    return {
        "ttft_s": ttft,
        "tokens_per_s": (n_new - 1) / (ttlt - ttft) if n_new > 1 else float("nan"),
        "total_s": end - start,
        "ttlt_s": ttlt,
        "new_tokens": n_new,
        "token_times": [t - start for t in timer.times],
        "answer": answer,
    }


speed_questions = eval_dataset.select(range(min(SPEED_N, len(eval_dataset))))["Question"]
speed_rows = []

for condition in CONDITIONS:
    with adapter_mode(condition):
        timed_answer(speed_questions[0], max_new_tokens=8)  # warm-up, not measured
        for question in speed_questions:
            result = timed_answer(question)
            speed_rows.append({"condition": condition, "question": question, **result})
    print(f"{condition}: {len(speed_questions)} answers timed")

speed_df = pd.DataFrame(speed_rows)
speed_df.drop(columns=["token_times"]).to_csv(f"{BENCH_DIR}/speed_per_question.csv", index=False)
display(speed_df.groupby("condition")[["ttft_s", "tokens_per_s", "total_s", "ttlt_s", "new_tokens"]].describe().T.round(3))

### 14.3 Precision test: MedQA and PubMedQA accuracy

For each question we build the prompt with the options, add `Answer:` at the start of the model turn and read the score of the next token for each option letter. The letter with the highest score is the prediction.

In [ ]:
MCQ_SYSTEM_PROMPT = "You are a medical expert. Choose the single best option."


def letter_ids(letter):
    # A letter can be tokenized as "A" or " A"; keep both ids.
    return {
        text_tokenizer.encode(letter, add_special_tokens=False)[-1],
        text_tokenizer.encode(" " + letter, add_special_tokens=False)[-1],
    }


@torch.inference_mode()
def pick_option(user_text, letters):
    messages = [
        {"role": "system", "content": MCQ_SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]
    prompt = apply_chat_template(messages, add_generation_prompt=True) + "Answer:"
    model_inputs = text_tokenizer(prompt, return_tensors="pt").to(model.device)
    logits = model(**model_inputs).logits[0, -1].float()
    scores = [max(logits[i].item() for i in letter_ids(letter)) for letter in letters]
    return letters[int(np.argmax(scores))]


def medqa_prompt(row):
    options = "\n".join(f"{letter}. {row['options'][letter]}" for letter in "ABCD")
    return (
        f"Question: {clean_text(row['question'])}\n\nOptions:\n{options}\n\n"
        "Answer with only the letter (A, B, C or D)."
    )


def pubmedqa_prompt(row):
    context = clean_text(" ".join(row["context"]["contexts"]))
    return (
        f"Context: {context}\n\nQuestion: {clean_text(row['question'])}\n\n"
        "Options:\nA. yes\nB. no\nC. maybe\n\nAnswer with only the letter (A, B or C)."
    )


medqa = load_dataset("GBaker/MedQA-USMLE-4-options", split="test").shuffle(seed=SEED)
medqa = medqa.select(range(min(MEDQA_N, len(medqa))))
pubmedqa = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train").shuffle(seed=SEED)
pubmedqa = pubmedqa.select(range(min(PUBMEDQA_N, len(pubmedqa))))
PUBMEDQA_LETTER = {"yes": "A", "no": "B", "maybe": "C"}

precision_rows = []
for condition in CONDITIONS:
    with adapter_mode(condition):
        for row in medqa:
            prediction = pick_option(medqa_prompt(row), list("ABCD"))
            precision_rows.append({"condition": condition, "benchmark": "MedQA", "question": row["question"],
                                   "prediction": prediction, "answer": row["answer_idx"]})
        for row in pubmedqa:
            prediction = pick_option(pubmedqa_prompt(row), list("ABC"))
            precision_rows.append({"condition": condition, "benchmark": "PubMedQA", "question": row["question"],
                                   "prediction": prediction, "answer": PUBMEDQA_LETTER[row["final_decision"]]})
    print(f"{condition}: {len(medqa)} MedQA + {len(pubmedqa)} PubMedQA questions scored")

precision_df = pd.DataFrame(precision_rows)
precision_df["correct"] = (precision_df["prediction"] == precision_df["answer"]).astype(float)
precision_df.to_csv(f"{BENCH_DIR}/precision_per_question.csv", index=False)
display(precision_df.pivot_table(index="benchmark", columns="condition", values="correct", aggfunc="mean").round(3))

### 14.4 Summary: metrics classified as speed or precision

One row per metric and condition, with the mean and the 95% interval. Saved to `benchmark_summary.csv` (with the model name) so the five notebooks can be compared later.

In [ ]:
SPEED_METRICS = [
    ("ttft_s", "TTFT (s)", "lower"),
    ("tokens_per_s", "Tokens per second", "higher"),
    ("total_s", "Total response time (s)", "lower"),
    ("ttlt_s", "TTLT (s)", "lower"),
]
PRECISION_METRICS = [
    ("MedQA", "MedQA accuracy", "higher"),
    ("PubMedQA", "PubMedQA accuracy", "higher"),
]

summary_rows = []
for condition in CONDITIONS:
    for column, label, better in SPEED_METRICS:
        values = speed_df[speed_df["condition"] == condition][column]
        mean, low, high = mean_ci(values)
        summary_rows.append({"model": MODEL_NAME, "category": "speed", "metric": label, "condition": condition,
                             "mean": mean, "ci_low": low, "ci_high": high, "better": better, "n": len(values)})
    for benchmark, label, better in PRECISION_METRICS:
        values = precision_df[(precision_df["condition"] == condition) & (precision_df["benchmark"] == benchmark)]["correct"]
        mean, low, high = mean_ci(values)
        summary_rows.append({"model": MODEL_NAME, "category": "precision", "metric": label, "condition": condition,
                             "mean": mean, "ci_low": max(low, 0.0), "ci_high": min(high, 1.0), "better": better, "n": len(values)})

bench_summary = pd.DataFrame(summary_rows)
bench_summary.to_csv(f"{BENCH_DIR}/benchmark_summary.csv", index=False)

table = bench_summary.copy()
table["value"] = table.apply(lambda r: f"{r['mean']:.3f} [{r['ci_low']:.3f}, {r['ci_high']:.3f}]", axis=1)
display(table.pivot_table(index=["category", "metric", "better"], columns="condition", values="value", aggfunc="first"))
print(f"Saved: {BENCH_DIR}/benchmark_summary.csv")

### 14.5 Plots

Each plot has its own figure. Error bars are the 95% interval; box plots show the spread over the questions.

#### Plot 1 - Token timeline (TTFT is where a line starts, the slope is tokens/s, TTLT is where it ends)

In [ ]:
fig, axes = plt.subplots(1, len(CONDITIONS), figsize=(11, 4), sharex=True, sharey=True)
for ax, condition in zip(axes, CONDITIONS):
    for times in speed_df[speed_df["condition"] == condition]["token_times"]:
        ax.plot(times, range(1, len(times) + 1), alpha=0.5)
    ax.set_title(f"{MODEL_NAME} - {condition}")
    ax.set_xlabel("seconds since the request")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("tokens generated")
plt.tight_layout()
plt.show()

#### Plot 2 - Speed metrics: mean and 95% interval

In [ ]:
def ci_bar(ax, rows, title, ylabel):
    yerr = [rows["mean"] - rows["ci_low"], rows["ci_high"] - rows["mean"]]
    ax.bar(rows["condition"], rows["mean"], yerr=yerr, capsize=6, color=["tab:gray", "tab:blue"][: len(rows)])
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.3)


fig, axes = plt.subplots(1, len(SPEED_METRICS), figsize=(14, 4))
for ax, (_, label, better) in zip(axes, SPEED_METRICS):
    rows = bench_summary[bench_summary["metric"] == label]
    ci_bar(ax, rows, f"{label}\n({better} is better)", "tokens / second" if "Tokens" in label else "seconds")
fig.suptitle(f"{MODEL_NAME} - speed")
plt.tight_layout()
plt.show()

#### Plot 3 - Speed metrics: spread over the questions (box plot)

In [ ]:
fig, axes = plt.subplots(1, len(SPEED_METRICS), figsize=(14, 4))
for ax, (column, label, _) in zip(axes, SPEED_METRICS):
    data = [speed_df[speed_df["condition"] == c][column].dropna() for c in CONDITIONS]
    ax.boxplot(data, showmeans=True)
    ax.set_xticks(range(1, len(CONDITIONS) + 1), CONDITIONS)
    ax.set_title(label)
    ax.grid(axis="y", alpha=0.3)
fig.suptitle(f"{MODEL_NAME} - speed spread")
plt.tight_layout()
plt.show()

#### Plot 4 - Precision: accuracy and 95% interval (dashed line = random guess)

In [ ]:
CHANCE = {"MedQA accuracy": 0.25, "PubMedQA accuracy": 1 / 3}

fig, axes = plt.subplots(1, len(PRECISION_METRICS), figsize=(9, 4), sharey=True)
for ax, (_, label, _) in zip(axes, PRECISION_METRICS):
    rows = bench_summary[bench_summary["metric"] == label]
    ci_bar(ax, rows, label, "accuracy")
    ax.axhline(CHANCE[label], color="red", linestyle="--", linewidth=1)
    ax.set_ylim(0, 1)
fig.suptitle(f"{MODEL_NAME} - precision")
plt.tight_layout()
plt.show()

#### Plot 5 - Speed vs precision (up and right is better)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
for condition, color in zip(CONDITIONS, ["tab:gray", "tab:blue"]):
    rows = bench_summary[bench_summary["condition"] == condition]
    speed = rows[rows["metric"] == "Tokens per second"]["mean"].iloc[0]
    precision = rows[rows["category"] == "precision"]["mean"].mean()
    ax.scatter(speed, precision, s=80, color=color)
    ax.annotate(condition, (speed, precision), textcoords="offset points", xytext=(6, 6))
ax.set_xlabel("tokens per second")
ax.set_ylabel("mean accuracy (MedQA, PubMedQA)")
ax.set_title(f"{MODEL_NAME} - speed vs precision")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()